# SIPTA Notebook: Validación de datos

Guía inicial para validar calidad y consistencia de los datasets crudos.

## Validación — Demografía

## Objetivos

- Revisar esquema y tipos de cada dataset.
- Identificar nulos, duplicados y valores fuera de rango.
- Confirmar existencia de identificador territorial por localidad.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

ruta_demografia = (
    RAW_DIR
    / "DEMOGRAFIA"
    / "osb_demografia-poblacion-localidad.csv"
)

df_poblacion = pd.read_csv(
    ruta_demografia,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

print(df_poblacion.shape)


(131502, 8)


## Funciones de validación

In [2]:
print("=== SEXO ===")
print(df_poblacion["SEXO"].value_counts())

print("\n=== EDADES ===")
print(f"Edad mínima: {df_poblacion['EDAD'].min()}")
print(f"Edad máxima: {df_poblacion['EDAD'].max()}")
print(f"Cantidad de edades distintas: {df_poblacion['EDAD'].nunique()}")

print("\n=== CURSO DE VIDA ===")
print(df_poblacion["CURSODEVIDA"].value_counts())

print("\n=== GRUPOS DE EDAD ===")
print(df_poblacion["GRUPOEDAD"].value_counts())

=== SEXO ===
SEXO
Hombres    65751
Mujeres    65751
Name: count, dtype: int64

=== EDADES ===
Edad mínima: 0
Edad máxima: 100
Cantidad de edades distintas: 101

=== CURSO DE VIDA ===
CURSODEVIDA
Vejez               53382
Adultez             40362
Juventud            14322
Infancia             7812
Primera Infancia     7812
Adolescencia         7812
Name: count, dtype: int64

=== GRUPOS DE EDAD ===
GRUPOEDAD
60 o más    53382
29 a 59     40362
00 a 11     15624
18 a 28     14322
12 a 17      7812
Name: count, dtype: int64


In [3]:
columnas_clave = [
    "ANO",
    "CODIGO_LOCALIDAD",
    "SEXO",
    "EDAD"
]

duplicados_clave = df_poblacion.duplicated(
    subset=columnas_clave
).sum()

print(
    "Duplicados según la clave "
    "ANO + LOCALIDAD + SEXO + EDAD:"
)
print(duplicados_clave)

Duplicados según la clave ANO + LOCALIDAD + SEXO + EDAD:
0


In [4]:
cobertura = (
    df_poblacion
    .groupby(["ANO", "CODIGO_LOCALIDAD"])
    .size()
    .reset_index(name="registros")
)

display(cobertura.head(30))

print("\n=== DISTRIBUCIÓN DE REGISTROS POR AÑO-LOCALIDAD ===")
print(cobertura["registros"].value_counts().sort_index())

,ANO,CODIGO_LOCALIDAD,registros
0,2005,0,202
1,2005,1,202
2,2005,2,202
3,2005,3,202
4,2005,4,202
5,2005,5,202
6,2005,6,202
7,2005,7,202
8,2005,8,202
9,2005,9,202



=== DISTRIBUCIÓN DE REGISTROS POR AÑO-LOCALIDAD ===
registros
202    651
Name: count, dtype: int64


In [6]:
print("=== VALIDACIÓN DE POBLACIÓN ===")

print("Población mínima:")
print(df_poblacion["POBLACION"].min())

print("\nPoblación máxima:")
print(df_poblacion["POBLACION"].max())

print("\nValores negativos:")
print((df_poblacion["POBLACION"] < 0).sum())

print("\nRegistros con población igual a 0:")
print((df_poblacion["POBLACION"] == 0).sum())

=== VALIDACIÓN DE POBLACIÓN ===
Población mínima:
0

Población máxima:
79199

Valores negativos:
0

Registros con población igual a 0:
526


In [7]:
print("=== EDAD MÍNIMA Y MÁXIMA POR GRUPO DE EDAD ===")

validacion_grupos = (
    df_poblacion
    .groupby("GRUPOEDAD")["EDAD"]
    .agg(["min", "max"])
    .sort_values("min")
)

display(validacion_grupos)

=== EDAD MÍNIMA Y MÁXIMA POR GRUPO DE EDAD ===


,min,max
GRUPOEDAD,,
00 a 11,0,11
12 a 17,12,17
18 a 28,18,28
29 a 59,29,59
60 o más,60,100


In [8]:
ejemplo = df_poblacion[
    (df_poblacion["ANO"] == 2024) &
    (df_poblacion["SEXO"] == "Hombres") &
    (df_poblacion["EDAD"] == 30)
]

bogota = ejemplo.loc[
    ejemplo["CODIGO_LOCALIDAD"] == 0,
    "POBLACION"
].sum()

localidades = ejemplo.loc[
    ejemplo["CODIGO_LOCALIDAD"] != 0,
    "POBLACION"
].sum()

print("Bogotá:", bogota)
print("Suma de localidades:", localidades)
print("Diferencia:", bogota - localidades)

Bogotá: 78564
Suma de localidades: 78564
Diferencia: 0


In [9]:
# Comparar el agregado Bogotá con la suma de las 20 localidades
# para todas las combinaciones de año, sexo y edad.

bogota = (
    df_poblacion[df_poblacion["CODIGO_LOCALIDAD"] == 0]
    [["ANO", "SEXO", "EDAD", "POBLACION"]]
    .rename(columns={"POBLACION": "POBLACION_BOGOTA"})
)

suma_localidades = (
    df_poblacion[df_poblacion["CODIGO_LOCALIDAD"] != 0]
    .groupby(["ANO", "SEXO", "EDAD"], as_index=False)["POBLACION"]
    .sum()
    .rename(columns={"POBLACION": "SUMA_LOCALIDADES"})
)

comparacion = bogota.merge(
    suma_localidades,
    on=["ANO", "SEXO", "EDAD"],
    how="inner"
)

comparacion["DIFERENCIA"] = (
    comparacion["POBLACION_BOGOTA"]
    - comparacion["SUMA_LOCALIDADES"]
)

print("=== COHERENCIA BOGOTÁ VS LOCALIDADES ===")
print(f"Comparaciones realizadas: {len(comparacion)}")
print(
    "Comparaciones con diferencia distinta de 0:",
    (comparacion["DIFERENCIA"] != 0).sum()
)
print(
    "Diferencia máxima absoluta:",
    comparacion["DIFERENCIA"].abs().max()
)

=== COHERENCIA BOGOTÁ VS LOCALIDADES ===
Comparaciones realizadas: 6262
Comparaciones con diferencia distinta de 0: 0
Diferencia máxima absoluta: 0


## Notas de validación — Demografía

- El dataset contiene 131.502 registros y 8 variables.
- La unidad territorial primaria `Localidad` está disponible mediante
  `CODIGO_LOCALIDAD` y `NOMBRE_LOCALIDAD`.
- Se identifican las 20 localidades de Bogotá más un agregado distrital
  (`CODIGO_LOCALIDAD = 0`, `NOMBRE_LOCALIDAD = Bogotá`).
- La serie temporal comprende 2005–2035.
- Los años posteriores al presente corresponden a proyecciones poblacionales
  y no deben interpretarse como observaciones realizadas.
- Cada combinación año-territorio contiene exactamente 202 registros,
  correspondientes a 2 sexos × 101 edades (0–100).
- No se encontraron valores nulos.
- No se encontraron duplicados exactos ni duplicados para la clave
  `ANO + CODIGO_LOCALIDAD + SEXO + EDAD`.
- No se encontraron valores negativos en `POBLACION`.
- Se identificaron 526 registros con población igual a cero; se conservan
  inicialmente al ser valores posibles en combinaciones de edad y territorio.
- Los intervalos de `GRUPOEDAD` son consistentes con la variable `EDAD`.
- En la comprobación realizada, el agregado distrital de Bogotá coincide
  exactamente con la suma de las 20 localidades.